# Constructing a dataset

In [1]:
import pandas as pd
import numpy as np

In [44]:
# load datasets
df_23 = pd.read_csv("data/raw/climate_misinfo_2023.csv")
df_24 = pd.read_csv("data/raw/climate_misinfo_2024.csv")
df_25 = pd.read_csv("data/raw/climate_misinfo_2025.csv")

In [45]:
# merge the datasets
df = pd.concat([df_23, df_24, df_25], ignore_index=True)

df.shape

(182584, 21)

In [46]:
# save full dataframe
df.to_csv("data/processed/climate_misinfo.csv", index=False)

## Create a stratified dataset for annotation testing

In [4]:
df = pd.read_csv("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/climate_misinfo.csv")

In [5]:
len(df)

182584

In [16]:
# set size of subset
TOTAL_SAMPLE = 10600
RANDOM_STATE = 42

EXCLUDE_KEYS = [
    "ea4d58f9",
    "ead59516",
    "e9abeb07",
    "e98df78c",
    "ea140444"
]

# filter out already annotated articles
df_filtered = df[~df["ArticleKey"].isin(EXCLUDE_KEYS)].copy()

print("Removed already annotated articles.")
print("Remaining rows:", len(df_filtered))


# compute year distribution
year_props = df_filtered["Year"].value_counts(normalize=True)

# compute target size per year
year_targets = (year_props * TOTAL_SAMPLE).round().astype(int)

sampled_list = []

Removed already annotated articles.
Remaining rows: 182579


In [17]:
# sample data
for year, year_n in year_targets.items():
    
    df_year = df_filtered[df_filtered["Year"] == year]
    
    # proportion per media within that year
    media_props = df_year["Media"].value_counts(normalize=True)
    media_targets = (media_props * year_n).round().astype(int)
    
    for media, media_n in media_targets.items():
        df_media = df_year[df_year["Media"] == media]
        
        # Undgå at sample mere end findes
        media_n = min(media_n, len(df_media))
        
        sampled_media = df_media.sample(
            n=media_n,
            random_state=RANDOM_STATE
        )
        
        sampled_list.append(sampled_media)

final_sample = pd.concat(sampled_list).reset_index(drop=True)

print("Final sample size:", final_sample.shape)
print(final_sample["Year"].value_counts(normalize=True))

Final sample size: (10068, 21)
Year
2023    0.416170
2024    0.324096
2025    0.259734
Name: proportion, dtype: float64


In [18]:
# check media distribution of sample
final_sample["Media"].value_counts(normalize=True)

Media
Ritzaus Bureau                       0.025129
Fyens.dk                             0.016488
Børsen.dk                            0.015296
Berlingske.dk                        0.014402
Altinget.dk                          0.014005
                                       ...   
KL.dk (Kommunernes Landsforening)    0.000099
Ktc.dk                               0.000099
Socialister.dk                       0.000099
Syddjursnetavis.dk                   0.000099
Amager Bladet                        0.000099
Name: proportion, Length: 814, dtype: float64

In [19]:
print("Count os unique media:", len(final_sample["Media"].unique()))

Count os unique media: 814


In [20]:
# save dataframe
final_sample.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/pilot_sample.parquet", index=False)